# 엔티티 메모리 — Store + Tool 패턴

## 이번 노트북 학습 목표

- 대화에서 인물·역할·프로젝트 정보를 추출하는 **엔티티 추출 체인(entity extraction chain)**을 만들어요.
- 추출한 정보를 세션별로 저장하는 **엔티티 스토어(entity store)** 패턴을 익혀요.
- 스토어의 정보를 프롬프트에 주입해 **개인화된 응답**을 만드는 방법을 이해해요.
- 구식 `ConversationEntityMemory` / `ConversationKGMemory`와의 차이를 비교해요.

## 사전 지식

- Modern 메모리 패턴 기초 (`ChatMessageHistory`, `RunnableWithMessageHistory`) (10번 노트북)
- 요약 기반 메모리 패턴 (13번 노트북)

## 구식 vs Modern 메모리 패턴 비교

| 항목 | 구식 (04번, `ConversationEntityMemory`) | Modern 엔티티 메모리 |
|------|----------------------------------------|---------------------|
| 메모리 클래스 | `ConversationEntityMemory(llm=...)` | 없음 (추출 체인 + 스토어로 대체) |
| 엔티티 추출 | 클래스 내부 자동 처리 | `entity_extract_chain` 함수로 명시적 처리 |
| 저장소 | 클래스 내부 dict | 세션별 외부 dict / DB로 분리 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

## 메모리 진화 다이어그램

```mermaid
flowchart LR
    subgraph OLD["구식 (04~06번)"]
        direction TB
        E1[ConversationEntityMemory]
        E2[ConversationKGMemory]
        E3[ConversationSummaryMemory]
    end

    subgraph NEW["Modern (14번)"]
        direction TB
        N1[엔티티 추출 체인<br/>entity_extract_chain]
        N2[엔티티 스토어<br/>entity_store dict / DB]
        N3[엔티티 인식 체인<br/>entity_aware_chain]
    end

    OLD -- "LangChain 1.0.x<br/>재설계" --> NEW

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class E1,E2,E3 error
    class N1,N3 process
    class N2 storage
```

> `ConversationEntityMemory`와 `ConversationKGMemory`(그래프 메모리)는 LangChain 0.2.x 이후 deprecated 상태예요. 이 노트북에서 배울 Store + Tool 패턴으로 같은 기능을 더 유연하게 구현할 수 있어요.

In [ ]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 세션별 대화 히스토리 (일반 대화용)
conversation_store: dict[str, ChatMessageHistory] = {}


def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in conversation_store:
        conversation_store[session_id] = ChatMessageHistory()
    return conversation_store[session_id]


# 세션별 엔티티 스토어 (아주 단순한 dict 기반 구조)
# 구조 예시: {"session_id": {"김철수": "프로젝트 A의 팀장" , ...}}
entity_store: dict[str, dict[str, str]] = {}


## 1. 엔티티 추출 체인 만들기

사용자 발화(utterance)에서 **인물·역할·프로젝트 정보를 요약한 텍스트**를 추출하는 체인을 만들어요. `entity_extract_chain`(엔티티 추출 체인)의 출력을 파싱해 세션별 딕셔너리에 저장하는 방식이에요.

실제 서비스에서는 JSON 구조로 파싱하는 것이 좋지만, 여기서는 개념 이해를 위해 사람이 읽기 좋은 한글 텍스트로만 정리해요.

### 엔티티 추출 흐름

```mermaid
flowchart TD
    UTTERANCE[사용자 발화] --> EXTRACT_PROMPT[entity_extract_prompt<br/>인물 / 역할 / 프로젝트 추출]
    EXTRACT_PROMPT --> LLM[ChatOpenAI]
    LLM --> SUMMARY[엔티티 요약 텍스트<br/>이름: 설명 형태]
    SUMMARY --> PARSE[줄 단위 파싱<br/>name, desc 분리]
    PARSE --> STORE[entity_store 갱신<br/>session_id 키로 저장]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class UTTERANCE input
    class EXTRACT_PROMPT,LLM,PARSE process
    class SUMMARY,STORE storage
```

> `entity_store`에는 **세션별로** 엔티티가 누적돼요. 같은 이름이 다시 등장하면 최신 정보로 덮어써요.

In [ ]:
# ============================================================
# TODO: 엔티티 추출 프롬프트와 update_entity_store 함수를 구현하세요
# 힌트: LLM이 발화에서 "이름: 설명" 형식으로 엔티티를 추출하도록 프롬프트를 작성하세요
# 예상 결과: "김철수는 팀장이에요" 발화에서 {'김철수': '팀장'} 형태로 저장되어야 합니다
# ============================================================

# ---------------------------------------------------
# 엔티티 추출 체인 및 스토어 업데이트 함수 구현
# ---------------------------------------------------

# 1단계: 엔티티 추출용 프롬프트 정의
# - 인물/역할/프로젝트 정보를 "이름: 설명" 형식으로 추출


# entity_extract_chain: 발화 → 엔티티 요약 텍스트 생성


def update_entity_store(session_id: str, utterance: str) -> str:
    """주어진 발화에서 엔티티 요약을 추출하고, 세션별 엔티티 스토어를 업데이트.

    - 줄 단위로 파싱해 "이름: 설명" 패턴만 저장
    - 같은 이름이 다시 나오면 최신 정보로 덮어씀
    """
    pass  # TODO: 구현하세요


## 2. 엔티티 메모리를 활용하는 답변 체인

세션별 엔티티 정보를 활용하여 **"누가 어떤 역할/프로젝트를 담당하는지"**를 기억하는 체인을 만들어요.

흐름:
1. 사용자 발화를 `ChatMessageHistory`(대화 이력 저장소)에 저장해요.
2. 같은 발화를 엔티티 추출 체인에 넣어 `entity_store`를 갱신해요.
3. 누적된 엔티티 정보를 `format_entity_summary()`로 텍스트화해 시스템 메시지로 LLM에 전달해요.

### 엔티티 메모리 체인 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문] --> HADD[history.add_user_message]
    QUESTION --> EXTRACT[update_entity_store<br/>엔티티 추출 + 저장]
    EXTRACT --> STORE[entity_store 갱신]
    STORE --> FMT[format_entity_summary<br/>엔티티 요약 텍스트]
    FMT --> PROMPT[entity_aware_prompt<br/>entity_summary + question]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> HADD2[history.add_ai_message]
    HADD2 --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class HADD,EXTRACT,FMT,PROMPT,LLM,HADD2 process
    class STORE storage
    class ANSWER output
```

> 엔티티 스토어는 **세션 내에서 계속 누적**돼요. 초기 발화에서 얻은 "김철수 = PM" 정보는 10번째 질문에서도 그대로 사용할 수 있어요.

In [ ]:
# ============================================================
# TODO: 엔티티 정보를 활용하는 답변 체인(entity_memory_chain)을 구현하세요
# 힌트: 발화에서 엔티티를 추출·저장한 뒤, format_entity_summary()로 프롬프트에 주입하세요
# 예상 결과: "김철수가 어떤 역할이었죠?" 질문에 스토어의 정보로 정확히 답해야 합니다
# ============================================================

# ---------------------------------------------------
# 엔티티 인식 답변 체인 및 메모리 통합 함수
# ---------------------------------------------------

# 1단계: 엔티티 정보를 활용하는 답변 프롬프트
# - entity_summary: format_entity_summary()가 생성하는 텍스트


# entity_aware_chain: 엔티티 요약을 시스템 메시지로 받아 답변 생성


def format_entity_summary(session_id: str) -> str:
    """세션의 엔티티 스토어를 LLM 프롬프트에 주입할 텍스트로 변환."""
    pass  # TODO: 구현하세요


def entity_memory_chain(inputs: dict) -> str:
    """엔티티 메모리를 활용하는 통합 체인 함수.

    실행 순서:
    1) 대화 히스토리에 사용자 발화 추가
    2) 발화에서 엔티티 추출 후 스토어 갱신
    3) 누적된 엔티티 정보를 텍스트로 변환해 프롬프트에 주입
    4) AI 응답을 대화 히스토리에 저장
    """
    pass  # TODO: 구현하세요


In [ ]:
# ---------------------------------------------------
# 엔티티 메모리 체인 테스트: 팀 구성 정보 누적 및 활용
# ---------------------------------------------------

session_id = "project_chat_01"

utterances = [
    "우리 팀에는 세 명이 있어요. 팀장은 김철수이고, 개발자는 박영희와 이민호입니다.",
    "김철수는 프로젝트 피닉스의 PM이고, 박영희는 프론트엔드, 이민호는 백엔드를 담당해요.",
    "이번 분기 안에 프로젝트 피닉스를 출시하는 것이 목표입니다.",
    "제가 팀 구성과 역할을 어떻게 설명드렸는지 요약해 주세요.",
]


## 핵심 요약

### 이번 노트북의 구성 요소

| 구성 요소 | 역할 |
|----------|------|
| `entity_extract_chain` | 발화에서 인물·역할·프로젝트 정보를 텍스트로 추출 |
| `update_entity_store(session_id, utterance)` | 추출 결과를 파싱해 세션별 딕셔너리에 저장 |
| `format_entity_summary(session_id)` | 딕셔너리를 LLM 프롬프트에 주입할 텍스트로 변환 |
| `entity_aware_chain` | 엔티티 요약을 시스템 메시지로 받아 답변 생성 |
| `entity_memory_chain` | 위 과정을 한 번에 묶는 체인 함수 |

### 구식 vs Modern 패턴 설계 비교

| 항목 | 구식 `ConversationEntityMemory` | Modern Store + Tool |
|------|--------------------------------|---------------------|
| 추출 로직 | 클래스 내부에 고정 | `entity_extract_chain` 함수로 교체 가능 |
| 저장 구조 | 클래스 내부 dict | 외부 dict / SQLite / VectorStore로 교체 가능 |
| 조회 방식 | 클래스 메서드 | `format_entity_summary()` 함수 (자유롭게 확장) |
| LangGraph 연동 | 어려움 | Tool로 래핑해 에이전트에 바로 연결 가능 |

### 주의사항

- 엔티티 추출마다 LLM 호출이 발생해요. 호출 빈도 조절(예: 중요 발화만 추출)로 비용을 줄일 수 있어요.
- 이 노트북의 스토어는 dict 기반이에요. 실제 서비스에서는 `pydantic`으로 스키마를 정의하고 DB에 저장해요.
- 복잡한 인물 관계(A가 B의 상사이고, B는 C 프로젝트 담당)는 그래프 데이터베이스(Neo4j 등)가 더 적합해요.
- 같은 이름의 엔티티가 반복되면 최신 정보로 덮어쓰기 때문에, 정보 충돌이 발생할 수 있어요.

### 실전 확장 방향

- `pydantic` 모델로 엔티티 구조를 정의하고 JSON 파싱으로 정확도를 높여요.
- 벡터 저장소(vector store)에 엔티티를 저장하면 유사 검색이 가능해져요.
- LangGraph 에이전트에서 `entity_memory_chain`을 도구(tool)로 등록해 자동으로 호출할 수 있어요.

### 다음 단계

이 노트북으로 04번 Memory 섹션이 마무리돼요. 다음 섹션에서는 여러 체인을 연결하는 **멀티 체인 패턴**과 **에이전트(Agent)** 구조를 학습해요.

---

## 연습 문제

### 연습 1: 회의록 기반 엔티티 메모리 + 액션 아이템 추적

엔티티 추출 체인을 확장하여, 대화에서 **인물 정보뿐만 아니라 액션 아이템(할 일)도 추적**하는 회의 어시스턴트를 만들어 보세요.

**요구사항:**
1. 기존 `entity_store` 외에 **`action_store`** (딕셔너리)를 추가하여, 세션별로 액션 아이템을 저장하세요.
2. 액션 아이템 추출용 프롬프트를 만드세요 (예: "아래 발화에서 누가 무엇을 해야 하는지 추출해줘. 형식: `담당자: 할일`").
3. 아래 회의 시나리오로 테스트하세요:
   - "이번 프로젝트에 김대리, 박과장, 최사원이 참여합니다."
   - "김대리는 다음 주까지 시장 조사 보고서를 작성해야 합니다."
   - "박과장은 예산안을 금요일까지 검토하고, 최사원은 경쟁사 분석을 맡았습니다."
   - "지금까지 정리된 팀원 정보와 각자의 할 일을 알려주세요."
4. 마지막 질문에서 **엔티티 정보와 액션 아이템을 모두 활용**한 응답이 나오는지 확인하세요.

**힌트:**
- 기존 `entity_extract_chain`과 유사한 구조로 `action_extract_chain`을 만드세요.
- 프롬프트에 엔티티 요약과 액션 아이템 목록을 함께 넣어서 LLM이 종합적으로 답변하도록 구성하세요.
- `entity_memory_chain` 함수를 참고하여 엔티티 + 액션을 모두 업데이트하는 새 체인 함수를 만드세요.

In [ ]:
# ============================================================
# TODO: 엔티티 + 액션 아이템을 함께 추적하는 회의 어시스턴트를 구현하세요
# 힌트: entity_store 외에 action_store를 추가하고, 각각 추출·저장 함수를 만드세요
# 예상 결과: 마지막 질문에서 팀원 정보와 할 일 목록을 함께 정리해야 합니다
# ============================================================

# ---------------------------------------------------
# 연습 1 풀이: 회의록 기반 엔티티 메모리 + 액션 아이템 추적
# ---------------------------------------------------

# 회의 시나리오 테스트
meeting_utterances = [
    "이번 프로젝트에 김대리, 박과장, 최사원이 참여합니다.",
    "김대리는 다음 주까지 시장 조사 보고서를 작성해야 합니다.",
    "박과장은 예산안을 금요일까지 검토하고, 최사원은 경쟁사 분석을 맡았습니다.",
    "지금까지 정리된 팀원 정보와 각자의 할 일을 알려주세요.",
]
